# 07 — From EDA to Recommendation

**Week 6: EDA → recommendation link** · Ahamed M.A.U (IT24101779) · Group AI-39

This notebook checks the **EDA predictions** (`logs/eda_insight_log.md`) against the **final model results** on the
untouched 2015/16 test season (`06_evaluation.ipynb`). It then looks at *where* the models are right and wrong, which
is what a coaching or analytics stakeholder needs in order to use the predictions. The written narrative built on these
numbers is `docs/eda_to_recommendation.md`.

It only reads saved outputs: no model is retrained and nothing here is used for model selection.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
OUTCOMES = ["Home Win", "Draw", "Away Win"]
MODELS = ["logistic_regression", "random_forest", "decision_tree", "knn", "majority_baseline"]

comparison = pd.read_csv("../outputs/tables/model_comparison.csv").set_index("model")
per_class = pd.read_csv("../outputs/tables/per_class_metrics.csv")
class_dist = pd.read_csv("../outputs/tables/class_distribution.csv")
predictions = pd.read_csv("../outputs/predictions/test_predictions.csv")
features = pd.read_csv("../data/processed/features.csv")

test = predictions.merge(features[["id", "Form_Diff", "Abs_Form_Diff"]], on="id", how="left")
assert test["Form_Diff"].notna().all() and len(test) == 3326
print("Test matches:", len(test))

Test matches: 3326


## 2. Final results (from `06_evaluation.ipynb`)

In [2]:
draw = per_class[per_class["class"] == "Draw"].set_index("model")[["precision", "recall", "f1"]].add_prefix("draw_")
summary = comparison[["accuracy", "macro_f1"]].join(draw).loc[MODELS]
summary.round(3)

,accuracy,macro_f1,draw_precision,draw_recall,draw_f1
model,,,,,
logistic_regression,0.455,0.434,0.287,0.281,0.284
random_forest,0.444,0.434,0.285,0.365,0.320
decision_tree,0.428,0.427,0.274,0.423,0.333
knn,0.431,0.389,0.264,0.185,0.217
majority_baseline,0.439,0.203,0.000,0.000,0.000


## 3. EDA predictions vs results

### 3.1 Home advantage is declining (EDA-09)

In [3]:
home_share = class_dist[class_dist["class"] == "Home Win"].set_index("split")["percentage"]
print(f"Home-win share: train {home_share['train']:.1f}% → test {home_share['test']:.1f}%")
print(f"Majority baseline accuracy on test: {comparison.loc['majority_baseline', 'accuracy']:.1%} "
      f"(45.9% on the full dataset in EDA)")

Home-win share: train 46.0% → test 43.9%
Majority baseline accuracy on test: 43.9% (45.9% on the full dataset in EDA)


### 3.2 Accuracy is misleading; draws need weighting (EDA-07, EDA-08)

Prediction mix vs actual mix on the test season: balanced class weights make the models predict every class, but not
always in the right proportions.

In [4]:
mix = pd.DataFrame({m: test[f"{m}_prediction"].value_counts(normalize=True) for m in MODELS[:4]})
mix["actual"] = test["actual_outcome"].value_counts(normalize=True)
(mix.loc[OUTCOMES] * 100).round(1)

,logistic_regression,random_forest,decision_tree,knn,actual
Home Win,37.4,33.7,28.2,51.9,43.9
Draw,25.2,32.9,39.7,18.0,25.7
Away Win,37.5,33.4,32.1,30.2,30.4


## 4. Where are the models right? Accuracy by pre-match form gap

EDA-25 found that outcomes are far more predictable when one team is in much better form. Does the final model
behave the same way?

In [5]:
gap_bins = pd.cut(test["Abs_Form_Diff"], [-1, 1, 3, 6, 15],
                  labels=["0–1 (even)", "2–3", "4–6", "7+ (mismatch)"])
rows = {}
for m in ["logistic_regression", "random_forest", "majority_baseline"]:
    correct = test[f"{m}_prediction"] == test["actual_outcome"]
    rows[m] = correct.groupby(gap_bins, observed=True).mean()
by_gap = pd.DataFrame(rows)
by_gap.insert(0, "matches", gap_bins.value_counts().sort_index())
by_gap.insert(1, "share_of_matches", (by_gap["matches"] / len(test)))
by_gap.to_csv("../outputs/tables/accuracy_by_form_gap.csv")
by_gap.round(3)

,matches,share_of_matches,logistic_regression,random_forest,majority_baseline
Abs_Form_Diff,,,,,
0–1 (even),825,0.248,0.419,0.393,0.406
2–3,987,0.297,0.402,0.382,0.461
4–6,974,0.293,0.455,0.459,0.443
7+ (mismatch),540,0.162,0.604,0.607,0.441


**Finding:**

- Accuracy rises from about **40%** for evenly matched teams to about **60%** when the form gap is 7+ points, for both
  top models. Against the baseline (44%), that is **+16 points** in mismatches.
- In **close games (gap ≤ 3, 55% of test matches) the models are no more accurate than always picking the home team**,
  and in the 2–3 band they are worse (40.2% / 38.2% vs 46.1%). Balanced weights make them predict draws and away wins in
  exactly these games, which raises Draw recall (and macro F1) but costs accuracy.
- The models' practical value is therefore concentrated in fixtures with a **clear form gap (4+ points, 45% of
  matches)**.

## 5. How do the models handle draws? (EDA-28)

In [6]:
draw_rows = {"actual draw rate": (test["actual_outcome"] == "Draw").groupby(gap_bins, observed=True).mean()}
for m in ["logistic_regression", "random_forest"]:
    draw_rows[f"{m} predicts Draw"] = (test[f"{m}_prediction"] == "Draw").groupby(gap_bins, observed=True).mean()
draws_by_gap = pd.DataFrame(draw_rows)
draws_by_gap.to_csv("../outputs/tables/draws_by_form_gap.csv")
(draws_by_gap * 100).round(1)

,actual draw rate,logistic_regression predicts Draw,random_forest predicts Draw
Abs_Form_Diff,,,
0–1 (even),28.5,41.3,54.9
2–3,24.1,31.5,44.5
4–6,26.8,16.8,17.9
7+ (mismatch),22.4,3.9,5.0


Prediction mix from Section 3.2: balanced weights also make the models **under-predict home wins** (LR 37.4%, RF 33.7%
vs 43.9% actual).

**Finding:** the models have learned "close match → draw". Logistic Regression predicts a draw in **41%** of
evenly-matched games, and Random Forest in **55%**, but only **28.5%** of those games actually end level. In mismatches
it is the reverse: they almost never predict a draw (4–5%), yet **22%** of those matches are draws. This matches EDA-28:
the draw rate hardly changes with form, so form-closeness is a weak draw signal, and the models lean on it too heavily.

## 6. Summary

| EDA finding | Prediction | Result on 2015/16 test season |
|---|---|---|
| EDA-09: home advantage declining | Test baseline below 45.9% | ✅ Home wins 43.9% of test matches (46.0% in training); baseline accuracy 43.9% |
| EDA-07: accuracy misleading | Macro F1 needed | ✅ Baseline 43.9% accuracy but 0.20 macro F1; best models 0.43 |
| EDA-08 / 28: draws hard, need weighting | Low Draw recall without weights | ✅ Unweighted LR/RF predicted ~0 draws (PR #6). Balanced weights: Draw recall 28–42%, still the weakest class |
| EDA-25: form gap carries signal | Easier to predict mismatches | ✅ Accuracy ~60% at a 7+ gap vs ~40% for even matches; in close games (≤ 3) no better than the home-win baseline |
| EDA-26: signal is weak overall (r = 0.27) | Modest macro F1 | ✅ Best macro F1 0.434: about double the baseline, far from perfect |
| EDA-28: closeness barely moves draw rate | Draw predictions unreliable | ✅ Models over-predict draws in close games (41–55% vs 28.5% actual) and miss them in mismatches |